# ClaimSense: explainable health-insurance claim adjudication
**Use case (from the use-case sheet, health insurance and claims):**
help claim assessors process reimbursement and cashless claims: read the discharge summary, check it against the policy
wording, compute what is payable, and draft the decision letter with clause references.

**The design principle:** *the LLM reads, the rules decide, a human signs off.* The model extracts clinical facts from free text;
deterministic Python applies waiting periods, exclusions, room-rent proportionate deduction, co-pay and sum-insured caps, with a
clause for every rupee; guardrails check the letter; every rejection, referral and high-value claim goes to a human.

Cells marked **LIVE** call OpenAI; everything else runs offline.

## 0. Setup
Runs in Google Colab or locally. In Colab the first cell clones the repo and installs this project's requirements; locally, start Jupyter from the project folder.

In [ ]:
import os, sys, subprocess
from pathlib import Path

PROJECT = "04-use-case-projects/claimsense-health-claims"
if "google.colab" in sys.modules:
    if not Path("genai-agentic-ai-training").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tarang16/genai-agentic-ai-training.git"], check=True)
    os.chdir(f"genai-agentic-ai-training/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path(PROJECT).exists():  # started from the repo root
    os.chdir(PROJECT)
sys.path.insert(0, os.getcwd())
print("Working in", os.getcwd())

In [ ]:
# LIVE: only needed for the cells marked LIVE. Never paste a key into a cell; getpass keeps it out of the file.
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

## 1. Policy wording and claims
A fictional policy written in the style of Indian retail health cover, and eight claims chosen to exercise each rule.

In [ ]:
import json, pandas as pd
from claimsense import PolicyWording
from claimsense.config import CLAIMS_DIR
policy = PolicyWording()
print(list(policy.clauses))
claims = {p.stem: json.loads(p.read_text()) for p in sorted(CLAIMS_DIR.glob("*.json"))}
pd.DataFrame([{"claim": k, "diagnosis_text": v["discharge_summary"][:90] + "...", "claimed": sum(i["amount"] for i in v["bill"])}
              for k, v in claims.items()])

## 2. Redaction

In [ ]:
from claimsense.privacy import redact
print(redact(claims["CLM-001"]["discharge_summary"]))

## 3. The rules engine, with no LLM
We feed it the *reference* facts a careful reviewer would extract. Look at CLM-003: the room was INR 8,000/day against an eligible 5,000 (1% of SI), so room, nursing, surgeon and OT are paid at 62.5%, but medicines and investigations are not cut.

In [ ]:
from claimsense import adjudicate
from claimsense.config import ROOT
ref = json.loads((ROOT / "data/eval/reference_facts.json").read_text())
a = adjudicate(claims["CLM-003"], ref["CLM-003"])
print(a.decision, a.payable, a.route)
pd.DataFrame([vars(l) for l in a.lines])

In [ ]:
for cid in claims:
    a = adjudicate(claims[cid], ref[cid])
    print(f"{cid}: {a.decision:8} payable={a.payable!s:>9}  route={a.route:13} clauses={[r['clause'] for r in a.reasons]}")

**Exercise:** CLM-006 is 16 days into the policy. Flip `is_accident` to `False` in its facts and re-run. Why does the decision change, and which clause is cited?

## 4. The full graph

In [ ]:
# LIVE
from claimsense import create_app, render
from IPython.display import Markdown
app = create_app()
print(app.get_graph().draw_mermaid())
state = app.invoke({"claim": claims["CLM-005"]})
Markdown(render(state))

In [ ]:
# LIVE: the ambiguous alcohol case must be referred, never auto-rejected
state = app.invoke({"claim": claims["CLM-007"]})
print(state["facts"]["alcohol_or_substance_related"], state["final"]["decision"], state["final"]["route"])

## 5. Evaluate
`python evaluate.py --offline` proves the rules (should be 100%). `python evaluate.py` measures the LLM's fact extraction against the reference and the end-to-end decision, payable and routing accuracy.

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "evaluate.py", "--offline"], capture_output=True, text=True).stdout)

## 6. Take it further
1. Add an ICD-10 coding step and a PED-to-ICD mapping instead of free-text PED matching.
2. Add a fraud signal: duplicate bills, admission on policy day 31, unusually high consumables ratio.
3. Load the real policy PDF (with permission) and keep the clause-id chunking.
4. Persist assessor overrides and use them as new golden cases.